# Graphs — Representation & Traversal

A **graph** $G=(V,E)$ is a set of **vertices** joined by **edges**, generalizing trees by allowing cycles, multiple paths, and disconnected pieces. Edges may be **directed** or not and may carry **weights**. The two standard machine representations trade space for lookup speed, and the two fundamental traversals — breadth-first and depth-first — underlie nearly every graph algorithm. Each traversal records snapshots so the expanding frontier can be watched moving across the graph.

$$ |E| \le \binom{|V|}{2} \text{ (undirected)}, \qquad \text{BFS/DFS} = O(|V| + |E|). $$

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from collections import deque
from IPython.display import display

plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def make_player(n_steps, render, label='step'):
    slider = widgets.IntSlider(value=0, min=0, max=n_steps-1, description=label,
                               continuous_update=False, layout=widgets.Layout(width='60%'))
    play = widgets.Play(value=0, min=0, max=n_steps-1, interval=650)
    widgets.jslink((play, 'value'), (slider, 'value'))
    out = widgets.interactive_output(render, {'k': slider})
    display(widgets.HBox([play, slider]), out)

# Undirected graph as adjacency lists.
ADJ = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F', 'G'],
    'F': ['C', 'E', 'G'],
    'G': ['E', 'F'],
}
VERTS = list(ADJ)
# fixed positions for stable drawing
COORD = {
    'A': (0, 2), 'B': (-1.5, 1), 'C': (1.5, 1),
    'D': (-2.5, 0), 'E': (-0.5, 0), 'F': (1.5, 0), 'G': (0.5, -1),
}

def draw_graph(visited=(), frontier=(), current=None, tree_edges=(), title='', order_label='', directed=False):
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    seen = set()
    for u in ADJ:
        for v in ADJ[u]:
            key = (u, v) if directed else tuple(sorted((u, v)))
            if key in seen: continue
            seen.add(key)
            x0, y0 = COORD[u]; x1, y1 = COORD[v]
            te = (u, v) in tree_edges or (v, u) in tree_edges
            ax.plot([x0, x1], [y0, y1], color='seagreen' if te else 'lightgray',
                    lw=3 if te else 1.4, zorder=1)
    for v in VERTS:
        x, y = COORD[v]
        if v == current:    c = 'tomato'
        elif v in visited:  c = 'seagreen'
        elif v in frontier: c = 'gold'
        else:               c = 'lightsteelblue'
        ax.add_patch(plt.Circle((x, y), 0.26, facecolor=c, edgecolor='k', zorder=2))
        ax.text(x, y, v, ha='center', va='center', fontsize=12, zorder=3)
    ax.text(0.02, 0.02, order_label, transform=ax.transAxes, fontsize=11, color='seagreen')
    xs=[p[0] for p in COORD.values()]; ys=[p[1] for p in COORD.values()]
    ax.set_xlim(min(xs)-0.7, max(xs)+0.7); ax.set_ylim(min(ys)-0.7, max(ys)+0.7)
    ax.set_title(title); ax.axis('off'); plt.show()

## Two Representations: Matrix vs. List

An **adjacency matrix** stores a $|V|\times|V|$ boolean grid — $O(1)$ edge lookup but $O(|V|^2)$ space, wasteful for sparse graphs. An **adjacency list** stores each vertex's neighbours — $O(|V|+|E|)$ space and ideal for sparse graphs, at the cost of $O(\deg)$ lookup. The widget shows both for our graph and highlights the cell/entry for a chosen edge.

$$ \text{matrix: } O(|V|^2) \text{ space}, \qquad \text{list: } O(|V|+|E|) \text{ space.} $$

In [2]:
def show_repr(u, v):
    n = len(VERTS); idx = {x:i for i,x in enumerate(VERTS)}
    M = np.zeros((n, n), dtype=int)
    for a in ADJ:
        for b in ADJ[a]: M[idx[a]][idx[b]] = 1
    connected = M[idx[u]][idx[v]] == 1
    fig, (axm, axl) = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={'width_ratios':[1.1,1]})
    axm.imshow(M, cmap='Blues', vmin=0, vmax=1)
    axm.set_xticks(range(n)); axm.set_xticklabels(VERTS)
    axm.set_yticks(range(n)); axm.set_yticklabels(VERTS)
    for i in range(n):
        for j in range(n):
            axm.text(j, i, M[i][j], ha='center', va='center',
                     color='white' if M[i][j] else 'gray', fontsize=9)
    axm.add_patch(plt.Rectangle((idx[v]-0.5, idx[u]-0.5), 1, 1, fill=False,
                  edgecolor='tomato', lw=3))
    axm.set_title(f'adjacency matrix  (M[{u}][{v}] = {M[idx[u]][idx[v]]})')
    axl.axis('off')
    for i, a in enumerate(VERTS):
        line = f'{a}: ' + ' -> '.join(ADJ[a])
        col = 'tomato' if a == u else 'black'
        axl.text(0, 1 - i*0.13, line, fontsize=11, color=col, family='monospace',
                 transform=axl.transAxes)
    axl.set_title('adjacency list')
    fig.suptitle(f'edge {u}-{v}: {"present" if connected else "absent"}')
    plt.tight_layout(); plt.show()

u_s = widgets.Dropdown(options=VERTS, value='B', description='u')
v_s = widgets.Dropdown(options=VERTS, value='E', description='v')
display(widgets.HBox([u_s, v_s]),
        widgets.interactive_output(show_repr, {'u': u_s, 'v': v_s}))

Output()

## Breadth-First Search — Expanding Rings

BFS explores from a source outward in layers using a **queue**, discovering all vertices at distance $d$ before any at $d+1$. On an unweighted graph this yields **shortest paths in edges**. The gold frontier is the queue; green edges form the BFS tree. Step through to watch concentric rings fill.

$$ \text{dist}(s, v) = \text{layer at which } v \text{ is first dequeued.} $$

In [3]:
def bfs_frames(adj, src):
    visited = set(); q = deque([src]); seen = {src}; tree = []
    frames = [(set(visited), list(q), None, list(tree), f'enqueue source {src}')]
    while q:
        node = q.popleft(); visited.add(node)
        frames.append((set(visited), list(q), node, list(tree), f'dequeue & visit {node}'))
        for nb in adj[node]:
            if nb not in seen:
                seen.add(nb); q.append(nb); tree.append((node, nb))
        if q:
            frames.append((set(visited), list(q), node, list(tree),
                           f'enqueue unseen neighbours of {node}: queue={list(q)}'))
    frames.append((set(visited), [], None, list(tree), 'done'))
    return frames

src_s = widgets.Dropdown(options=VERTS, value='A', description='source')
bfs_area = widgets.Output()
def relaunch_bfs(*_):
    frames = bfs_frames(ADJ, src_s.value)
    order = []
    def draw(k):
        visited, frontier, current, tree, note = frames[k]
        vis_now = visited - ({current} if current else set())
        if current and current not in order: order.append(current)
        seq = [v for v in order]
        draw_graph(visited=vis_now, frontier=set(frontier), current=current, tree_edges=tree,
                   title=f'BFS step {k}/{len(frames)-1}: {note}',
                   order_label='visited: ' + ' '.join(seq))
    bfs_area.clear_output(wait=True)
    with bfs_area: make_player(len(frames), draw)
src_s.observe(relaunch_bfs, 'value')
display(src_s, bfs_area)
relaunch_bfs()

Dropdown(description='source', options=('A', 'B', 'C', 'D', 'E', 'F', 'G'), value='A')

Output()

## Depth-First Search — Plunge and Backtrack

DFS follows one path as deep as possible before backtracking, driven by a **stack** (here explicit). It does not find shortest paths but reveals structure: connectivity, cycles, and topological order. Compare its visit order to BFS on the same source — same vertices, very different sequence.

$$ \text{DFS visits a vertex, then recurses on the first unvisited neighbour.} $$

In [4]:
def dfs_frames(adj, src):
    visited = []; stack = [src]; seen = {src}; tree = []; parent = {src: None}
    frames = [(list(visited), list(stack), None, list(tree), f'push source {src}')]
    while stack:
        node = stack.pop()
        if node in visited: continue
        visited.append(node)
        frames.append((list(visited), list(stack), node, list(tree), f'pop & visit {node}'))
        pushed = []
        for nb in reversed(adj[node]):       # reversed so first neighbour ends on top
            if nb not in seen:
                seen.add(nb); stack.append(nb); pushed.append(nb)
                tree.append((node, nb))
        if pushed:
            frames.append((list(visited), list(stack), node, list(tree),
                           f'push unvisited neighbours: stack={list(stack)}'))
    frames.append((list(visited), [], None, list(tree), 'done'))
    return frames

dsrc_s = widgets.Dropdown(options=VERTS, value='A', description='source')
dfs_area = widgets.Output()
def relaunch_dfs(*_):
    frames = dfs_frames(ADJ, dsrc_s.value)
    def draw(k):
        visited, stack, current, tree, note = frames[k]
        vis_now = set(visited) - ({current} if current else set())
        draw_graph(visited=vis_now, frontier=set(stack), current=current, tree_edges=tree,
                   title=f'DFS step {k}/{len(frames)-1}: {note}',
                   order_label='visited: ' + ' '.join(visited))
    dfs_area.clear_output(wait=True)
    with dfs_area: make_player(len(frames), draw)
dsrc_s.observe(relaunch_dfs, 'value')
display(dsrc_s, dfs_area)
relaunch_dfs()

Dropdown(description='source', options=('A', 'B', 'C', 'D', 'E', 'F', 'G'), value='A')

Output()

## Connected Components via Repeated Traversal

Running a traversal from every not-yet-visited vertex partitions the graph into **connected components** — maximal sets of mutually reachable vertices. This is the basis of "is the graph connected?" and of union-style grouping. The widget adds a disconnected piece and colors each component as it is discovered.

$$ \text{components} = \text{number of traversal restarts needed to cover } V. $$

In [5]:
ADJ2 = {**{k: list(v) for k, v in ADJ.items()}, 'H': ['I'], 'I': ['H', 'J'], 'J': ['I']}
COORD2 = {**COORD, 'H': (3, 1.2), 'I': (3.6, 0.4), 'J': (3, -0.4)}
VERTS2 = list(ADJ2)

def components_frames(adj):
    comp_of = {}; comp_id = 0; frames = []
    for start in adj:
        if start in comp_of: continue
        comp_id += 1
        stack = [start]
        while stack:
            n = stack.pop()
            if n in comp_of: continue
            comp_of[n] = comp_id
            frames.append((dict(comp_of), n, comp_id, f'component {comp_id}: add {n}'))
            for nb in adj[n]:
                if nb not in comp_of: stack.append(nb)
    frames.append((dict(comp_of), None, comp_id, f'{comp_id} components found'))
    return frames

frames_cc = components_frames(ADJ2)
PALETTE = ['lightsteelblue', 'gold', 'salmon', 'mediumaquamarine', 'plum']
def draw_cc(k):
    comp_of, current, ncomp, note = frames_cc[k]
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    seen=set()
    for u in ADJ2:
        for v in ADJ2[u]:
            key=tuple(sorted((u,v)))
            if key in seen: continue
            seen.add(key)
            x0,y0=COORD2[u]; x1,y1=COORD2[v]
            ax.plot([x0,x1],[y0,y1], color='lightgray', lw=1.4, zorder=1)
    for v in VERTS2:
        x,y = COORD2[v]
        if v == current: c='tomato'
        elif v in comp_of: c=PALETTE[(comp_of[v]-1) % len(PALETTE)]
        else: c='whitesmoke'
        ax.add_patch(plt.Circle((x,y),0.26, facecolor=c, edgecolor='k', zorder=2))
        ax.text(x,y,v, ha='center', va='center', fontsize=12, zorder=3)
    xs=[p[0] for p in COORD2.values()]; ys=[p[1] for p in COORD2.values()]
    ax.set_xlim(min(xs)-0.7,max(xs)+0.7); ax.set_ylim(min(ys)-0.7,max(ys)+0.7)
    ax.set_title(f'components step {k}/{len(frames_cc)-1}: {note}'); ax.axis('off'); plt.show()
make_player(len(frames_cc), draw_cc)

Output()